In [0]:
%run ./config

In [0]:
repayments_df=spark.read.csv(silver+'/repayments.csv', header=True, inferSchema=True)

In [0]:
loans_df=spark.read.csv(silver+'/loans.csv', header=True, inferSchema=True)

In [0]:
loan_status_events_df=spark.read.csv(silver+'/loan_status_events.csv', header=True, inferSchema=True)

In [0]:
borrower_profile_df=spark.read.csv(silver+'/borrower_profile.csv', header=True, inferSchema=True)

In [0]:
joined_df=borrower_profile_df.join(loans_df, on='borrower_id', how='inner')\
                             .join(loan_status_events_df, on='loan_id', how='inner')\
                             .join(repayments_df, on='loan_id', how='inner')

In [0]:
joined_df.write.mode('overwrite').csv(silver+'/joined_fact_table.csv', header=True)
df=spark.read.csv(silver+'/joined_fact_table.csv', header=True, inferSchema=True)

In [0]:
df.createOrReplaceTempView('df')
fact_table=spark.sql("""
                     with cte1 as (
                         select loan_id,
                                borrower_id, badge, urs_score, 
                                branch_id,  
                                principal,
                                sum(amount_paid) amount_paid
                    from df 
                    group by loan_id,borrower_id, branch_id, principal, badge,urs_score
                     ),
                     cte2 as (select    loan_id , 
                                        borrower_id, badge, urs_score, 
                                        branch_id,
                                        amount_paid, principal,
                                        principal-amount_paid as outstanding_amount 
                              from cte1)
                    select  c2.loan_id,
                            c2.borrower_id, c2.badge,c2.urs_score, 
                            c2.branch_id, 
                            c2.principal, 
                            round(c2.outstanding_amount,2) outstanding_amount
                    from cte2 c2 

                     """)
fact_table.write.format('delta').option("overwriteSchema", "true").mode('overwrite').save(gold+'/fact_table')